# Shape Analysis | Project 1 - Primitive Fitting

In this project, you will implement a shape abstraction method that fits a set of simple geometric primitives to the surface of an input mesh.\
To do this, you will first implement a local feature descriptor whose features can then be used to extract regions (clusters of faces) that belong to the same primitive.\
Finally, you will fit a parametric cylinder to each cluster of faces and visualize the resulting set of cylinders.

There is no single solution to this problem. You are encouraged to experiment with design choices and parameters. However, you should aim for a meaningful abstraction of the given meshes, and we expect you to justify your design choices in the presentation.\
We provide three example meshes to test your implementation. You can find them in the data folder.

**Some important notes:**

- Make sure your final submission notebook can correctly process and visualize all three meshes, as well as the intermediate results (shape diameter function, regions before and after merging, ...).
- Which mesh gets processed and visualized is controlled by the `MODEL` variable at the top of the notebook. Also define your hyperparameters in the corresponding match section.\
  We will use this variable to test your implementation on the three meshes for grading.
- The notebook should run without errors and terminate in a reasonable amount of time.
- **If you have any questions, come to the tutorials!**

# Example: Initial Regions, After Merging, Cylinders
![image.png](image.png)

## Setup

The most important libraries for this project are numpy, trimesh, and k3d. Read the documentation for these libraries to understand how to use them. You can install them using pip.
You are also allowed to use scipy.

**Important: Do not use any other external libraries for solving the project. You are allowed to use the default libraries of Python. For visualization, please use k3d.**

Below is the match statement to help you manage the parameters for different meshes.

*Hint: You may need to normalize and preprocess your mesh before you can effectively apply the cylinder fitting. Inspect your intermediate results (e.g. from region growing) with k3d.*

In [106]:
import numpy as np
import trimesh as tm
import k3d

# Optional
import math
import scipy
from queue import Queue # You can use all queues from queue module!
from matplotlib import pyplot as plt

In [107]:
# You can use the following functions as helpers to visualize the mesh
# You can also use them in other parts of your code if needed!


def rotation_matrix_from_vectors(vec1, vec2):
    ''' Compute the rotation matrix to rotate vec1 to vec2. '''

    a = vec1 / np.linalg.norm(vec1)
    b = vec2 / np.linalg.norm(vec2)

    v = np.cross(a, b) #perpendicular to a and b, rotation axis
    c = np.dot(a, b)   # cos
    s = np.linalg.norm(v)

    K = np.array([
        [0, -v[2], v[1]], 
        [v[2], 0, -v[0]],
        [-v[1], v[0], 0],
    ])

    R = np.eye(3) + K + K @ K * (1 - c) / (s ** 2)

    return R


def get_cylinder_mesh(centroid, axis, height, radius):
    ''' The centroid lies at the center of the cylinder. Therefore, it extends by height/2 in both directions. '''

    cylinder = tm.creation.cylinder(radius=radius, height=height, sections=16)

    R = rotation_matrix_from_vectors(np.array([0, 0, 1]), axis)
    cylinder.vertices = cylinder.vertices @ R.T
    cylinder.vertices += centroid

    return cylinder

In [ ]:
MODEL = 'spot'

match MODEL:

    case 'spot':
        FILEPATH = 'data/spot.obj',
        NUM_RAYS = 12,
        CONE_ANGLE = 90
        ## your spot parameters...

    case 'lion':
        FILEPATH = 'data/lion.obj',
        NUM_RAYS = 12,
        CONE_ANGLE = 60 #narrow cone for faster changing shape surfaces
        ## your lion parameters...

    case 'penguin':
        FILEPATH = 'data/penguin.obj',
        NUM_RAYS = 12,
        CONE_ANGLE = 90
        ## your penguin parameters...

    case _:
        raise ValueError(f"Unknown model: {MODEL}")
    
## Example usage: mesh = tm.load(FILEPATH)

In [136]:
## def load_and_preprocess_mesh(filepath, ...)

def load_and_preprocess_mesh(model, filepath): 

    mesh = tm.load(filepath)

    #center the mesh, mean of vertices becomes  the center 
    mesh.vertices = mesh.vertices - mesh.vertices.mean(0) # not good if vertices are not evenly distributed 
    mesh.vertices = mesh.vertices - (mesh.vertices.max(axis=0) + mesh.vertices.min(axis=0)) / 2 # not good if we have even a single outlier

    #scale mesh, so that it is inside the unit sphere
    mesh.vertices = mesh.vertices / np.linalg.norm(mesh.vertices, axis=-1).max()    

    #rotating the models, so the aminals are looking in negative y direction 
    match model:

        case 'spot':
            #rotate around z-axis with 180 degree
            rot_z = tm.transformations.rotation_matrix(np.radians(180), [0, 0, 1])
            #rotate around x-axis with 90 degree
            rot_x = tm.transformations.rotation_matrix(np.radians(-90), [1, 0, 0])

            combined_matrix_spot = rot_x @ rot_z
            mesh.apply_transform(combined_matrix_spot)

        case 'lion': 
            #rotate around x-axis with 90 degree
            mesh.apply_transform(tm.transformations.rotation_matrix(np.radians(90), [1, 0, 0])) 

        case 'penguin':
            #rotate around x-axis with 90 degree
            mesh.apply_transform(tm.transformations.rotation_matrix(np.radians(90), [1, 0, 0]))  

        case _:
            raise ValueError(f"Unknown model: {MODEL}")
        
    #!!!!!maybe we don't need to return anything 
    return mesh.vertices, mesh.faces



spot_v, spot_f = load_and_preprocess_mesh(model = 'spot', filepath='data/spot.obj')
lion_v, lion_f = load_and_preprocess_mesh(model = 'lion', filepath='data/lion.obj')
penguin_v, penguin_f = load_and_preprocess_mesh(model = 'penguin', filepath='data/penguin.obj')

plot = k3d.plot()
plot += k3d.mesh(spot_v, spot_f, color=0xff0000, opacity=0.5)
plot += k3d.mesh(lion_v, lion_f, color=0x00ff00, opacity=0.5)
plot += k3d.mesh(penguin_v, penguin_f, color=0x0000ff, opacity=0.5)
display(plot)

#just a new way to test the resulting orientation
"""
# Define the paths and colors mapped to your MODEL names
models_config = {
    'spot':    {'path': 'data/spot.obj',    'color': 0xff0000},
    'lion':    {'path': 'data/lion.obj',    'color': 0x00ff00},
    'penguin': {'path': 'data/penguin.obj', 'color': 0x0000ff}
}

plot = k3d.plot()
processed_data = {}

# The loop explicitly unpacks 'MODEL' (the key) and 'config' (the values)
for model, config in models_config.items():
    
    # Pass both the path AND the current MODEL name string into the function
    v, f = load_and_preprocess_mesh(model, config['path'])
    
    # Save arrays dynamically into a dictionary using the current MODEL name
    processed_data[f"{model}_v"] = v
    processed_data[f"{model}_f"] = f
    
    # Render onto our unified plot
    plot += k3d.mesh(v, f, color=config['color'], opacity=0.5)

display(plot)
"""


C:\Users\Startklar\AppData\Roaming\Python\Python311\site-packages\traittypes\traittypes.py:98: UserWarning: Given trait value dtype "float64" does not match required type "float32". A coerced copy has been created.
  warnings.warn(
C:\Users\Startklar\AppData\Roaming\Python\Python311\site-packages\traittypes\traittypes.py:98: UserWarning: Given trait value dtype "int64" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(


Plot(antialias=3, axes=['x', 'y', 'z'], axes_helper=1.0, axes_helper_colors=[16711680, 65280, 255], background…

'\n# Define the paths and colors mapped to your MODEL names\nmodels_config = {\n    \'spot\':    {\'path\': \'data/spot.obj\',    \'color\': 0xff0000},\n    \'lion\':    {\'path\': \'data/lion.obj\',    \'color\': 0x00ff00},\n    \'penguin\': {\'path\': \'data/penguin.obj\', \'color\': 0x0000ff}\n}\n\nplot = k3d.plot()\nprocessed_data = {}\n\n# The loop explicitly unpacks \'MODEL\' (the key) and \'config\' (the values)\nfor model, config in models_config.items():\n\n    # Pass both the path AND the current MODEL name string into the function\n    v, f = load_and_preprocess_mesh(model, config[\'path\'])\n\n    # Save arrays dynamically into a dictionary using the current MODEL name\n    processed_data[f"{model}_v"] = v\n    processed_data[f"{model}_f"] = f\n\n    # Render onto our unified plot\n    plot += k3d.mesh(v, f, color=config[\'color\'], opacity=0.5)\n\ndisplay(plot)\n'

## Descriptor Function

There are many possible local feature descriptors that can be used to extract regions of the mesh. In this project, you will implement the shape diameter function as presented in the lecture.\
The function should take a mesh as input and return an array of shape diameter values, one value for each vertex. You may want to pass other parameters to the function, such as the number of rays used to sample the value of each vertex.\
Note that different meshes may require different parameter settings. Use the match statement to define the parameters for each mesh and use these variables in your function call:
```python
diam_values = compute_shape_diameter_function(mesh, num_rays, important_parameter=IMPORTANT_PARAMETER) # Define IMPORTANT_PARAMETER in the match statement
```

Because the shape diameter function returns a value for each vertex, but region growing is performed on the faces, you will also need to compute a per-face descriptor based on your vertex descriptors.\
There are multiple ways to do this. Try out different ones to see which one works best for the region growing task.

*Hint: trimesh provides a function to compute ray mesh intersections. If you additionally pip install pyembree/embreex, you can speed up the ray casting significantly!*\
*Hint: You may want to experiment with the log-alpha-normalization for a better region growing!*

**For your submission: use k3d to visualize both your vertex and face descriptors!**

In [ ]:
#to be deleted, only for visualisation 
mesh = tm.load('data/spot.obj', force='mesh')
plot = k3d.plot()

# 1. Render the solid mesh
plot += k3d.mesh(mesh.vertices, mesh.faces, color=0x00ffff, opacity=0.8) 

inwards_vectors = mesh.vertex_normals * -1
# 2. Render the normal vectors as arrows
# We pass the starting points (vertices) and the direction vectors (vertex_normals)
plot += k3d.vectors(
    origins=mesh.vertices,
    vectors=inwards_vectors * 0.05,  # Multiply by 0.05 so the arrows aren't too long
    color=0xff0000
)

display(plot)

Plot(antialias=3, axes=['x', 'y', 'z'], axes_helper=1.0, axes_helper_colors=[16711680, 65280, 255], background…

In [ ]:
## def compute_shape_diameter_function(mesh, ...) 

def sample_cone_vectors(center_direction, cone_angle_rad, num_rays):
    """
    Generates a localized cone of unit vectors around a specific center_direction.
    """
    # 1. Generate random directions over a spherical cap pointing up along +Z
    cos_theta_max = np.cos(cone_angle_rad / 2.0)
    
    # Uniformly sample azimuthal angle (phi) and polar height (cos_theta)
    phi = np.random.uniform(0, 2 * np.pi, num_rays)
    cos_theta = np.random.uniform(cos_theta_max, 1.0, num_rays)
    sin_theta = np.sqrt(1.0 - cos_theta**2)
    
    # Standard spherical to cartesian conversion (assuming Z is up)
    local_rays = np.stack([
        sin_theta * np.cos(phi),
        sin_theta * np.sin(phi),
        cos_theta
    ], axis=1) # Shape: (num_rays, 3)
    
    # 2. Find the rotation matrix that transforms the Z-axis [0, 0, 1] 
    # into our targeted center_direction
    center_direction = center_direction / np.linalg.norm(center_direction)
    z_axis = np.array([0.0, 0.0, 1.0])
    
    # If the direction is already pointing along Z, no transformation is needed
    if np.allclose(center_direction, z_axis):
        return local_rays
    elif np.allclose(center_direction, -z_axis):
        return local_rays * np.array([1, 1, -1])
        
    # Standard Rodrigues' rotation formula or trimesh utility to find alignment matrix
    rotation_matrix = tm.geometry.align_vectors(z_axis, center_direction)
    
    # 3. Rotate our localized cone vectors to point along our vertex direction
    # rotation_matrix is a 4x4 matrix, we extract the top left 3x3 rotation block
    world_rays = local_rays @ rotation_matrix[:3, :3].T
    
    return world_rays

    # Ensure direction vector is normalized
    inwards_vectors = mesh.vertex_normals * -1
    inwards_vectors =  inwards_vectors/ np.linalg.norm(inwards_vectors)



#other parameters might be 
def compute_shape_diameter_function(mesh, ...):

    #1.define a cone in negative normal direction   





    #2. shoot rays 

    #3. record distances to intersections 

    distance,_,_ =  tm.ray.intersect_location(ray_origins= mesh.vertices    , ray_directions = <array of random directions>)

    #4. filter recorded distances 
    #4.1 reject outside intersections (intersection has a normal pointing in the same direction as the origin point of the ray)
    #4.2 remove distances that are more than 1sigma from the median distance

SyntaxError: incomplete input (2458280000.py, line 14)

In [ ]:
## Visualize per vertex features

In [ ]:
## def vertex_descriptor_to_face_descriptor(mesh, vertex_descriptor, ...)

In [ ]:
## Visualize per face features

## Region Growing

Implement the region growing algorithm as presented in the lecture. The function should take a mesh and an array of per-face descriptors as input and return a list of clusters, where each cluster is a list of face indices. You may want to pass other parameters to the function, such as thresholds required for growing. Start growing from seed faces with local maximum features. Think of reasonable heuristics to decide when to stop growing a cluster and which faces to add.\
Since region growing may return a large number of clusters, you should also implement a function that merges clusters into neighboring clusters, if they are similar enough.\
Also here, think of reasonable heuristics to decide when to merge adjacent clusters.

*Hint: For the merging step, you can look at the volume of the convex hull of the individual clusters versus the possibly combined clusters to decide whether to merge them.*\
*Hint: trimesh provides several useful functions to compute properties of meshes (e.g. convex hull or volume). You can apply them to submeshes as well.*\
*Hint: Carefully read the documentation if you want to use trimesh functions. They may require properties your submeshes do not have.*

**For your submission: use k3d to visualize the clusters before and after merging!**

In [ ]:
## def grow_regions(mesh, face_features, ...)

In [ ]:
## Visualize initial regions

In [ ]:
## def merge_regions(mesh, clusters, ...)

In [ ]:
## Visualize merged regions

## Cylinder Fitting

Now that you have a set of clusters, you can fit a cylinder to each cluster. Implement the cylinder fitting algorithm as presented in the lecture. **Do not use any other primitives!**\
The function `fit_cylinder` should take a (sub)mesh as input and return suitable parameters `(centroid, axis, height, radius)` for a cylinder.\
Similarly, the function `fit_cylinders` should take a mesh and a set of clusters as input and return a list of cylinder parameters, one for each cluster.

*Hint: For the shape operator, you may want to consider a larger neighborhood per vertex (with suitable weighting) for robust results.*\
*Hint: You can use the functions provided in the top of the notebook for your cylinder fitting and visualization.*

**For your submission: use k3d to visualize the fitted cylinders!**

In [ ]:
## def fit_cylinder(mesh, ...)

In [ ]:
## def fit_cylinders(mesh, clusters, ...)

In [ ]:
## Use a similar approach to this to visualize the cylinders:
##
## cylinders = fit_cylinders(mesh, clusters, ...)
##
## plot = k3d.plot()
## plot.add(mesh)
##
## for (centroid, axis, height, radius) in cylinders:
##    cylinder_mesh = get_cylinder_mesh(centroid, axis, height, radius)
##    plot.add(cylinder_mesh)
##
## display(plot)